# Build a project-specific verb

Domain add-ons are usually projects that define their own verbs. This vignette
implements a heat-stress vocabulary without modifying or registering anything
inside CubeDynamics.

## A verb factory

The outer function captures configuration. The inner `_op` receives the current
pipe value and returns the next value. Here the return type is a Dataset with an
occurrence state and an exceedance magnitude.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v


def heat_stress(*, threshold: float = 35.0):
    """Classify temperature values at or above ``threshold``."""
    def _op(cube: xr.DataArray) -> xr.Dataset:
        if "time" not in cube.dims:
            raise ValueError("heat_stress requires a 'time' dimension")

        state = (cube >= threshold).rename("state")
        magnitude = (cube - threshold).where(state, 0).rename("magnitude")
        result = xr.Dataset({"state": state, "magnitude": magnitude})
        result.attrs.update(cube.attrs)
        result.attrs.update(project_verb="heat_stress", threshold=float(threshold))
        return result

    return _op

## Compose shared and project vocabularies

In [ ]:
time = pd.date_range("2025-07-01", periods=5, freq="D")
temperature = xr.DataArray(
    np.array(
        [
            [[32.0, 34.0], [33.0, 35.0]],
            [[33.0, 35.0], [34.0, 36.0]],
            [[35.0, 37.0], [36.0, 38.0]],
            [[34.0, 36.0], [35.0, 37.0]],
            [[31.0, 33.0], [32.0, 34.0]],
        ]
    ),
    dims=("time", "y", "x"),
    coords={"time": time, "y": [1, 0], "x": [0, 1]},
    name="air_temperature",
    attrs={"units": "degC", "source": "deterministic synthetic vignette"},
)

states = (pipe(temperature) | heat_stress(threshold=35.0)).unwrap()
daily_fraction = (
    pipe(states["state"])
    | v.mean(dim=("y", "x"), keep_dim=False)
).unwrap()

assert set(states.data_vars) == {"state", "magnitude"}
assert states.attrs["threshold"] == 35.0
assert daily_fraction.dims == ("time",)
daily_fraction.to_dataframe(name="heat_stress_fraction")

## Test direct use and pipe use

In [ ]:
direct = heat_stress(threshold=35.0)(temperature)
through_pipe = (pipe(temperature) | heat_stress(threshold=35.0)).unwrap()
xr.testing.assert_identical(direct, through_pipe)
print("The project verb has the same result in direct and pipe use.")

In a real add-on, move `heat_stress` into `my_project.verbs`, document the
threshold's scientific meaning, and keep this regression test with the project.
The repository's `examples/custom_verb_project/` directory provides that small
module layout.